# AtliQ Bank — End-to-End Banking Data Analysis & Customer Insights

<h1 style="font-family: 'Poppins'; font-weight: bold; color: green;">👨‍💻 Author: Bibi Rehana</h1>

[![GitHub](https://img.shields.io/badge/GitHub-Profile-blue?style=for-the-badge&logo=github)](https://github.com/Rehana5210/)  
[![Kaggle](https://img.shields.io/badge/Kaggle-Profile-blue?style=for-the-badge&logo=kaggle)](https://www.kaggle.com/bibirehana)  
[![LinkedIn](https://img.shields.io/badge/LinkedIn-Profile-blue?style=for-the-badge&logo=linkedin)](https://www.linkedin.com/in/rehana-rehana-9a37782b9/)


## 1. Business Objective

AtliQ Bank wants to understand whether its campaign improved customer activity. The central question is whether the difference in transactions between the control and test groups is meaningful and statistically supported.

This analysis combines customer, credit, and transaction data to answer business questions about:
- customer demographics and financial behavior
- credit profile and risk indicators
- campaign performance and business impact


## 2. Data Science Questions

### Primary question
Is there a statistically significant difference between average transactions in the control and test groups?

### Supporting questions
1. How large is the difference between the two groups?
2. What percentage uplift did the test group achieve?
3. Are there missing values or duplicates?
4. Are there unusual or abnormal observations?
5. How are transaction values distributed?
6. How did transactions change over time?
7. Was the test group consistently higher than the control group?
8. Is the observed campaign effect statistically significant?
9. What is the estimated business impact?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

possible_dirs = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "math_&_statistics",
    Path("c:/Users/datalab/OneDrive/Documents/my_projects"),
    Path("c:/Users/datalab/OneDrive/Documents/my_projects/math_&_statistics"),
]

for d in possible_dirs:
    if d.exists():
        base_dir = d
        break
else:
    base_dir = Path.cwd()

if (base_dir / "math_&_statistics").exists():
    data_dir = base_dir / "math_&_statistics"
else:
    data_dir = base_dir

print(f"Using data directory: {data_dir}")


In [ ]:
customer_path = data_dir / "customers.csv"
credit_path = data_dir / "credit_profiles.csv"
transaction_path = data_dir / "avg_transactions_after_campaign.csv"

df_cust = pd.read_csv(customer_path)
df_cr = pd.read_csv(credit_path)
df_tr = pd.read_csv(transaction_path)

print("customer shape:", df_cust.shape)
print("credit shape:", df_cr.shape)
print("transaction shape:", df_tr.shape)

print("Customer nulls:")
print(df_cust.isnull().sum())
print("Credit nulls:")
print(df_cr.isnull().sum())
print("Transaction nulls:")
print(df_tr.isnull().sum())

print("Customer duplicates:", df_cust.duplicated().sum())
print("Credit duplicates:", df_cr.duplicated().sum())
print("Transaction duplicates:", df_tr.duplicated().sum())

df_cust.head()


## 3. Data Overview

The project includes three datasets:
- Customer data: demographic and behavioral attributes for 1,000 customers
- Credit profile data: financial risk and credit metrics for customer-level records
- Transaction data: daily average transaction values for the control and test groups over the campaign period

This structure allows us to connect customer characteristics to campaign outcomes and credit behavior while validating the main campaign metric at a daily level.


In [ ]:
# Check missing income patterns
print(df_cust[df_cust["annual_income"].isna()].head())
print("Missing annual income count:", df_cust["annual_income"].isna().sum())

occupation_median_income = df_cust.groupby("occupation")["annual_income"].median()
print("Occupation-wise median income:")
print(occupation_median_income.sort_values())

occupation_med_map = df_cust.groupby("occupation")["annual_income"].median().to_dict()

df_cust["annual_income"] = df_cust["annual_income"].fillna(df_cust["occupation"].map(occupation_med_map))
overall_income_median = df_cust["annual_income"].median()
df_cust["annual_income"] = df_cust["annual_income"].fillna(overall_income_median)

print("Remaining missing income values:", df_cust["annual_income"].isna().sum())

df_cust.isnull().sum()


In [ ]:
# Age quality check and outlier handling
print("Age summary:")
print(df_cust["age"].describe())
print("Age values outside plausible range:")
print(df_cust[(df_cust["age"] < 15) | (df_cust["age"] > 100)][["cust_id", "age", "occupation"]].head())

valid_age_mask = df_cust["age"].between(15, 100)
valid_age_median = df_cust.loc[valid_age_mask, "age"].median()

for idx, val in df_cust[~valid_age_mask]["age"].items():
    df_cust.at[idx, "age"] = valid_age_median

print("Adjusted age summary:")
print(df_cust["age"].describe())
print("Remaining impossible ages:", ((df_cust["age"] < 15) | (df_cust["age"] > 100)).sum())


## 4. Customer Profile Analysis

The customer dataset gives the foundation for segmentation. It shows the bank's customer mix across locations, genders, occupations, and income levels. Understanding these patterns helps explain who responded most strongly to the campaign.


In [ ]:
# Customer descriptive stats and income distribution
customer_summary = df_cust.describe(include="all").T
print(customer_summary)

plt.figure(figsize=(10, 5))
sns.histplot(df_cust["annual_income"], bins=30, kde=True, color="green")
plt.title("Distribution of Annual Income")
plt.xlabel("Annual Income")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

occupation_income = df_cust.groupby("occupation")["annual_income"].mean().sort_values(ascending=False)
plt.figure(figsize=(12, 6))
occupation_income.plot(kind="bar", color="steelblue")
plt.title("Average Annual Income by Occupation")
plt.xlabel("Occupation")
plt.ylabel("Average Annual Income")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Demographic views
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df_cust, x="gender", ax=axes[0], palette="Set2")
axes[0].set_title("Customer Distribution by Gender")

sns.countplot(data=df_cust, x="location", ax=axes[1], palette="Set3")
axes[1].set_title("Customer Distribution by Location")

sns.countplot(data=df_cust, x="marital_status", ax=axes[2], palette="viridis")
axes[2].set_title("Customer Distribution by Marital Status")

plt.tight_layout()
plt.show()

bins = [0, 18, 35, 50, 65, 100]
labels = ["0-17", "18-34", "35-49", "50-64", "65+"]
df_cust["age_group"] = pd.cut(df_cust["age"], bins=bins, labels=labels, right=False, include_lowest=True)
age_counts = df_cust["age_group"].value_counts().sort_index()

plt.figure(figsize=(8, 8))
plt.pie(age_counts.values, labels=age_counts.index, autopct="%1.1f%%", startangle=90, colors=["#FF9999", "#66B3FF", "#99FF99", "#FFCC99", "#C2C2F0"])
plt.title("Customer Distribution by Age Group")
plt.axis("equal")
plt.tight_layout()
plt.show()


## 5. Credit Profile Analysis

The credit dataset helps us understand financial behavior beyond transaction activity. It includes debt, utilization, inquiries, and credit limits, which are important for risk assessment and customer segmentation.


In [ ]:
# Credit profile cleaning
if df_cr["cust_id"].duplicated().any():
    df_cr = df_cr.drop_duplicates(subset="cust_id", keep="first").reset_index(drop=True)

numeric_cols = df_cr.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    df_cr[col] = df_cr[col].fillna(df_cr[col].median())

print(df_cr.isnull().sum())
print("Final credit profile shape:", df_cr.shape)
print(df_cr.describe().T)


In [ ]:
# Credit score distribution
plt.figure(figsize=(10, 5))
plt.hist(df_cr["credit_score"].dropna(), bins=30, color="purple", edgecolor="black")
plt.title("Distribution of Credit Scores")
plt.xlabel("Credit Score")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_cr, x="credit_utilisation", y="credit_limit", hue="credit_score", palette="viridis")
plt.title("Credit Utilization vs Credit Limit")
plt.xlabel("Credit Utilisation")
plt.ylabel("Credit Limit")
plt.tight_layout()
plt.show()


## 6. Transaction Data Quality & Campaign Evaluation

The transaction table is the main evidence for the campaign result. Before comparing groups statistically, we validate date consistency, missing values, and duplicates. This ensures the final conclusion is based on clean data.


In [ ]:
# Validate transaction table
df_tr = pd.read_csv(transaction_path)
df_tr["campaign_date"] = pd.to_datetime(df_tr["campaign_date"], errors="coerce")
df_tr = df_tr.sort_values("campaign_date").reset_index(drop=True)

print("Missing values before cleaning:")
print(df_tr.isnull().sum())
print("Duplicate dates:", df_tr["campaign_date"].duplicated().sum())

full_date_range = pd.date_range(start=df_tr["campaign_date"].min(), end=df_tr["campaign_date"].max(), freq="D")
missing_dates = full_date_range.difference(df_tr["campaign_date"])
print("Missing daily dates:", len(missing_dates))

if len(missing_dates) > 0:
    missing_df = pd.DataFrame({"campaign_date": missing_dates})
    df_tr = pd.concat([df_tr, missing_df], ignore_index=True).sort_values("campaign_date").reset_index(drop=True)

for col in ["control_group_avg_tran", "test_group_avg_tran"]:
    df_tr[col] = pd.to_numeric(df_tr[col], errors="coerce")
    df_tr[col] = df_tr[col].interpolate(method="time", limit_direction="both")
    df_tr[col] = df_tr[col].fillna(df_tr[col].median())

print("Missing values after cleaning:")
print(df_tr.isnull().sum())
print("Transaction summary:")
print(df_tr.describe())

df_tr.head()


In [ ]:
# Daily transaction comparison
plt.figure(figsize=(12, 6))
plt.plot(df_tr["campaign_date"], df_tr["control_group_avg_tran"], label="Control", color="steelblue", linewidth=2)
plt.plot(df_tr["campaign_date"], df_tr["test_group_avg_tran"], label="Test", color="darkorange", linewidth=2)
plt.title("Average Transactions Over Time: Control vs Test Group")
plt.xlabel("Campaign Date")
plt.ylabel("Average Transactions")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
box_data = pd.concat([
    df_tr["control_group_avg_tran"].rename("Control"),
    df_tr["test_group_avg_tran"].rename("Test")
], axis=1)

sns.boxplot(data=box_data, palette="Set2")
plt.title("Transaction Distribution by Group")
plt.ylabel("Average Transactions")
plt.tight_layout()
plt.show()


In [ ]:
# Campaign effectiveness metrics
control_mean = df_tr["control_group_avg_tran"].mean()
test_mean = df_tr["test_group_avg_tran"].mean()
mean_difference = test_mean - control_mean
uplift_pct = (mean_difference / control_mean) * 100

print(f"Control group mean: {control_mean:.2f}")
print(f"Test group mean: {test_mean:.2f}")
print(f"Mean difference: {mean_difference:.2f}")
print(f"Uplift: {uplift_pct:.2f}%")

stat, p_value = stats.ttest_rel(df_tr["control_group_avg_tran"], df_tr["test_group_avg_tran"])
print(f"Paired t-test statistic: {stat:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("Decision: the campaign effect is statistically significant at the 5% level.")
else:
    print("Decision: the campaign effect is not statistically significant at the 5% level.")


## 7. Data Science Questions — Answers

### Q1. Is there a statistically significant difference between average transactions in the control and test groups?
Yes, if the p-value from the paired t-test is below 0.05. The campaign effect can then be considered statistically significant.

### Q2. How large is the difference between the two groups?
The difference is measured as the mean of the test group minus the mean of the control group.

### Q3. What percentage uplift did the test group achieve?
The uplift percentage is calculated as the difference divided by the control mean, multiplied by 100.

### Q4. Are there missing values or duplicates?
The notebook checks for nulls and duplicate dates or customer IDs before running comparisons. Missing values are resolved using a domain-aware method instead of simply dropping rows.

### Q5. Are there unusual or abnormal observations?
Age and income values were checked for unrealistic ranges. This is important because extreme values can skew the analysis and distort the real story.

### Q6. How are transactions distributed?
The distribution is reviewed through descriptive statistics and boxplots.

### Q7. How did transactions change over time?
The line chart shows the daily movement of both groups during the campaign period.

### Q8. Was the test group consistently higher than the control group?
This is answered visually by comparing the daily curves and average uplift.

### Q9. What is the estimated business impact?
The total incremental transaction impact is estimated by multiplying the average uplift by the number of days in the observation period.


## 8. Conclusion & Business Recommendation

The analysis suggests the campaign had a positive effect on customer activity. The test group showed stronger transaction performance than the control group and the observed difference can be evaluated through statistical evidence.

### Recommended actions
- continue the campaign in a similar or slightly refined form
- monitor whether the uplift is strongest among certain customer segments
- expand the campaign only after validating results on a larger sample or in the next campaign cycle
- use customer profile and credit profile insights to target the users most likely to respond


## 9. Final Visualization Suggestions

To present the result professionally to stakeholders, the final dashboard or report should include:
1. A bar chart comparing control vs test average transactions
2. A time-series line chart for daily group performance
3. A boxplot showing transaction distribution by group
4. A small summary callout with uplift percentage and p-value
5. A segmented chart by customer type or location if campaign performance varies by segment

These visuals turn the analysis into a business-ready story and make the effect easy to understand at a glance.


In [ ]:
# Final business impact estimate
avg_daily_delta = df_tr["test_group_avg_tran"].mean() - df_tr["control_group_avg_tran"].mean()
total_period_days = len(df_tr)
estimated_incremental_transactions = avg_daily_delta * total_period_days

print(f"Average daily uplift: {avg_daily_delta:.2f}")
print(f"Estimated incremental transactions over the period: {estimated_incremental_transactions:.2f}")

plt.figure(figsize=(6, 5))
plt.bar(["Control", "Test"], [control_mean, test_mean], color=["#4C78A8", "#F58518"])
plt.title("Average Transactions by Group")
plt.ylabel("Average Transactions")
plt.tight_layout()
plt.show()
